In [0]:
# Cell 1 — read clean silver table and confirm
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver = spark.table("raw.silver.cms_claims_clean")

print(f"Rows: {df_silver.count():,}")
print(f"Columns: {len(df_silver.columns)}")
print("\nSample columns:")
df_silver.select("provider_id", "specialty", "procedure_code", 
                  "avg_submitted_charge", "avg_medicare_paid").show(3, truncate=False)

Rows: 1,999,992
Columns: 30

Sample columns:
+-----------+-----------------+--------------+--------------------+-----------------+
|provider_id|specialty        |procedure_code|avg_submitted_charge|avg_medicare_paid|
+-----------+-----------------+--------------+--------------------+-----------------+
|1003000126 |Internal Medicine|99217         |288.93477273        |58.619772727     |
|1003000126 |Internal Medicine|99219         |424.80411765        |109.15529412     |
|1003000126 |Internal Medicine|99220         |686.56428571        |151.59685714     |
+-----------+-----------------+--------------+--------------------+-----------------+
only showing top 3 rows


In [0]:
# Cell 2 — add feature columns
from pyspark.sql.window import Window

# Window partitioned by specialty
w_specialty = Window.partitionBy("specialty")

df_features = df_silver \
    .withColumn("charge_to_payment_ratio",
        F.when(F.col("avg_medicare_paid") == 0, None)
         .otherwise(F.col("avg_submitted_charge") / F.col("avg_medicare_paid"))) \
    .withColumn("billing_intensity",
        F.when(F.col("total_beneficiaries") == 0, None)
         .otherwise(F.col("total_services") / F.col("total_beneficiaries"))) \
    .withColumn("specialty_avg_charge",
        F.avg("avg_submitted_charge").over(w_specialty)) \
    .withColumn("specialty_stddev_charge",
        F.stddev("avg_submitted_charge").over(w_specialty)) \
    .withColumn("provider_charge_zscore",
        F.when(F.col("specialty_stddev_charge") == 0, 0)
         .otherwise(
            (F.col("avg_submitted_charge") - F.col("specialty_avg_charge")) /
             F.col("specialty_stddev_charge"))) \
    .withColumn("facility_flag",
        F.when(F.col("place_of_service") == "F", 1).otherwise(0)) \
    .withColumn("features_created_at",
        F.current_timestamp())

print(f"Rows: {df_features.count():,}")
print(f"Columns: {len(df_features.columns)}")
print("\nNew feature columns:")
df_features.select(
    "provider_id", "specialty",
    "charge_to_payment_ratio", "billing_intensity",
    "provider_charge_zscore", "facility_flag"
).show(3, truncate=False)

Rows: 1,999,992
Columns: 37

New feature columns:
+-----------+----------+-----------------------+------------------+----------------------+-------------+
|provider_id|specialty |charge_to_payment_ratio|billing_intensity |provider_charge_zscore|facility_flag|
+-----------+----------+-----------------------+------------------+----------------------+-------------+
|1003019571 |Hematology|3.1931225053730428     |1.0               |0.04445715681912794   |0            |
|1003019571 |Hematology|3.8143070449300938     |1.0               |0.6764439927823317    |1            |
|1003019571 |Hematology|3.5646729388550886     |1.0677966101694916|-0.3979336283551147   |0            |
+-----------+----------+-----------------------+------------------+----------------------+-------------+
only showing top 3 rows


In [0]:
# Cell 3 — write features table
df_features.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("raw.silver.cms_claims_features")

print(f"Features table written: {df_features.count():,} rows")
print(f"Columns: {len(df_features.columns)}")

Features table written: 1,999,992 rows
Columns: 37


In [0]:
# Cell 4 — verify
spark.sql("SHOW TABLES IN raw.silver").show()

print("\nFeature stats:")
df_features.select(
    "charge_to_payment_ratio",
    "billing_intensity",
    "provider_charge_zscore",
    "facility_flag"
).describe().show()

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
|  silver|   cms_claims_clean|      false|
|  silver|cms_claims_features|      false|
+--------+-------------------+-----------+


Feature stats:
+-------+-----------------------+------------------+----------------------+-------------------+
|summary|charge_to_payment_ratio| billing_intensity|provider_charge_zscore|      facility_flag|
+-------+-----------------------+------------------+----------------------+-------------------+
|  count|                1999972|           1999992|               1999992|            1999992|
|   mean|      6.167977170926519|  4.79677889289573|  4.104459592062683...|0.36227144908579634|
| stddev|       84.3988820406019|209.51948458565914|    0.9999744995601134|0.48065680248906945|
|    min|    0.38035961272475793|0.6760869565217391|   -2.1857938394544623|                  0|
|    max|                50000.0|204812.93333333332| 